# Microsoft GraphRAG — a step-by-step quickstart notebook

This notebook walks you through Microsoft's **GraphRAG** from zero: what it is, then installing it, indexing a book (*A Christmas Carol*), asking questions, and poking around inside the knowledge graph it builds.

**Run the cells top to bottom.** Each step explains what it does and why before you run it.

> ⚠️ **Cost warning:** indexing makes a *lot* of LLM calls. This notebook uses cheap "mini" models and a short book so a run should cost very little — but check your OpenAI usage page afterwards so you know what to expect before indexing anything bigger.

## Part 1 — What GraphRAG is (the 2-minute version)

**Ordinary RAG** (Retrieval-Augmented Generation): chop documents into chunks → store them in a vector database → when a question comes in, fetch the most *similar* chunks → hand them to the LLM to answer.

That's great for *needle-in-a-haystack* questions ("what does clause 7 say?"), but bad at *whole-collection* questions ("what are the main themes across all of this?"), because no single chunk contains that answer.

**GraphRAG** fixes this in two stages:

1. **Indexing** (expensive, done once)
   - An LLM reads every chunk and extracts **entities** (people, organisations, places, events) and the **relationships** between them → a **knowledge graph**.
   - A community-detection algorithm (**Leiden**) groups closely connected entities into **communities**, at several zoom levels.
   - The LLM writes a **summary report** for each community.
2. **Querying** (done many times) — you pick a search mode:
   - **Global** — big-picture questions. Every community summary answers the question in parallel, then the partial answers are merged (map-reduce).
   - **Local** — questions about specific entities. Starts at matching entities and walks their neighbourhood.
   - **DRIFT** — a hybrid of the two.
   - **Basic** — plain vector RAG, handy as a baseline for comparison.

**One line:** ordinary RAG finds *similar text*; GraphRAG builds a *map of who/what is connected*, summarises each region of the map, and answers from the map.

## Part 2 — Before you start (one-off setup, done in a terminal)

GraphRAG needs **Python 3.10, 3.11 or 3.12** (3.13 may break it). It's best to run this notebook inside its own **virtual environment** (a self-contained Python just for this project).

In a terminal, from the folder containing this notebook:

**Mac / Linux**
```bash
python3.12 -m venv .venv          # or python3.11 / python3.10
source .venv/bin/activate
pip install -r requirements.txt
python -m ipykernel install --user --name graphrag --display-name "Python (graphrag)"
jupyter lab                       # or: jupyter notebook
```

**Windows (PowerShell)**
```powershell
py -3.12 -m venv .venv
.venv\Scripts\activate
pip install -r requirements.txt
python -m ipykernel install --user --name graphrag --display-name "Python (graphrag)"
jupyter lab
```

Then, in Jupyter, choose **Kernel → Change kernel → Python (graphrag)**.

You also need an **OpenAI API key** with a little credit on it (https://platform.openai.com/api-keys). You'll paste it in at Step 4 — it is *not* stored in this notebook.

## Step 1 — Check your Python version

The cell below prints which Python the notebook is running and stops with a clear message if it's outside 3.10–3.12.

*Python refresher:* `sys.version_info` is a tuple like `(3, 12, 4, ...)`, and tuples compare item by item, so `(3, 10) <= (3, 12) < (3, 13)` works as a range check.

In [ ]:
import sys

print("Python executable:", sys.executable)
print("Python version:   ", sys.version.split()[0])

assert (3, 10) <= sys.version_info[:2] < (3, 13), (
    "GraphRAG needs Python 3.10–3.12. Create the venv with one of those versions "
    "(see Part 2) and switch this notebook's kernel to it."
)
print("✅ Python version is OK")

## Step 2 — Install GraphRAG (skip if you used `requirements.txt`)

`%pip install` installs into *the same Python this notebook is running* — safer than `!pip`, which might hit a different Python. It takes a minute or two because GraphRAG has a lot of dependencies.

> 💡 If Jupyter tells you to restart the kernel after installing, do so, then carry on from the next cell. Restarting wipes everything the notebook has run so far — imports and variables — so after *any* restart, re-run the cells from here downwards in order.

In [ ]:
%pip install -q graphrag pandas pyarrow networkx matplotlib

### A small helper to run the `graphrag` command

GraphRAG is mainly driven from the command line (`graphrag init`, `graphrag index`, `graphrag query`). Rather than relying on your terminal's PATH, this helper finds the `graphrag` program that sits next to this notebook's Python and runs it, printing the output as it goes.

*Python refresher:*
- `pathlib.Path` is the modern way to handle file paths (works on Windows and Mac/Linux).
- `subprocess.Popen` starts another program; we read its output line by line so you see progress live.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Folder holding this Python's scripts: .venv/bin (Mac/Linux) or .venv\Scripts (Windows)
SCRIPTS_DIR = Path(sys.executable).parent
GRAPHRAG = shutil.which("graphrag", path=str(SCRIPTS_DIR)) or shutil.which("graphrag")
assert GRAPHRAG, "Couldn't find the graphrag command — did Step 2 install successfully?"
print("Using:", GRAPHRAG)


def run(*args):
    """Run `graphrag <args>` and stream its output into the notebook."""
    cmd = [GRAPHRAG, *args]
    print("$", " ".join(cmd), "\n")
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        encoding="utf-8", errors="replace",
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"graphrag exited with code {proc.returncode} — read the output above.")


run("--help")

## Step 3 — Initialise a GraphRAG project

`graphrag init` creates a project folder containing:

| Item | What it is |
|---|---|
| `input/` | where your text files go |
| `.env` | holds your API key (`GRAPHRAG_API_KEY=...`) |
| `settings.yaml` | every pipeline setting — models, chunk sizes, entity types, etc. |
| `prompts/` | the prompt templates GraphRAG sends to the LLM (worth reading!) |

We pass the models as flags so it doesn't stop to ask:
- `-m gpt-4.1-mini` — a cheap chat model for extraction, summaries and answers
- `-e text-embedding-3-small` — a cheap embedding model

Change `PROJECT` if you'd like the project somewhere else.

In [ ]:
PROJECT = Path("christmas").resolve()

if (PROJECT / "settings.yaml").exists():
    print(f"Project already initialised at {PROJECT} — skipping. "
          "(Delete the folder to start again from scratch.)")
else:
    run("init", "--root", str(PROJECT), "-m", "gpt-4.1-mini", "-e", "text-embedding-3-small")

print("\nProject contents:")
for p in sorted(PROJECT.iterdir()):
    print("  ", p.name + ("/" if p.is_dir() else ""))

## Step 4 — Add your OpenAI API key

`getpass` shows a password-style box so the key never appears on screen or gets saved inside the notebook. The cell writes it into the project's `.env` file.

> 🔒 `.env` is listed in `.gitignore`, so it won't be pushed to GitHub. Never commit API keys.

In [ ]:
from getpass import getpass

env_file = PROJECT / ".env"
key = getpass("Paste your OpenAI API key (input is hidden): ").strip()
assert key.startswith("sk-"), "That doesn't look like an OpenAI key (they start with 'sk-')."

env_file.write_text(f"GRAPHRAG_API_KEY={key}\n", encoding="utf-8")
del key  # don't keep it hanging around in memory
print("✅ Key written to", env_file)

## Step 5 — Download the sample book

*A Christmas Carol* from Project Gutenberg — short enough to keep costs low, rich enough in characters and relationships to make the graph interesting.

We use Python's built-in `urllib`, so this works the same on Windows, Mac and Linux.

In [ ]:
import urllib.request

book = PROJECT / "input" / "book.txt"
book.parent.mkdir(parents=True, exist_ok=True)

if not book.exists():
    url = "https://www.gutenberg.org/cache/epub/24022/pg24022.txt"
    urllib.request.urlretrieve(url, book)

text = book.read_text(encoding="utf-8")
print(f"{book.name}: {len(text):,} characters, ~{len(text.split()):,} words\n")
print(text[:600])

## Step 6 (optional) — Peek at the settings

`settings.yaml` controls everything. Things worth noticing:
- the **model** blocks (which models are used, and the API key reference)
- **`chunks`** — how the text is cut up
- **`extract_graph` → `entity_types`** — what kinds of entities the LLM looks for (default: organisation, person, geo, event). Changing these to suit your domain is one of the biggest quality levers.

The cell just prints the file; edit it in any text editor if you want to change something.

In [ ]:
print((PROJECT / "settings.yaml").read_text(encoding="utf-8"))

## Step 7 — Build the index (this is the expensive bit)

`graphrag index` runs the whole pipeline: chunk → extract entities & relationships → build the graph → detect communities → write community reports → embed text.

Expect **several minutes**. You'll see each workflow tick by. When it finishes there'll be an `output/` folder full of `.parquet` files (a compact table format pandas can read).

**If it fails**, it's nearly always one of:
- a wrong API key (re-run Step 4),
- no credit on your OpenAI account,
- a rate limit — just re-run; GraphRAG caches finished LLM calls, so it picks up where it left off rather than paying twice.

💡 *Cheaper alternative:* change `"standard"` to `"fast"` below. The fast method uses classic NLP instead of the LLM for parts of extraction — lower quality, much lower cost.

In [ ]:
run("index", "--root", str(PROJECT), "--method", "standard")

## Step 8 — Ask questions

A tiny wrapper so asking is one line. The query text goes first, then the method.

In [ ]:
def ask(question, method="global"):
    """Ask GraphRAG a question. method: 'global', 'local', 'drift' or 'basic'."""
    run("query", question, "--root", str(PROJECT), "--method", method)

### 8a — Global search: a big-picture question
Uses the community summaries, map-reduce style. This is where GraphRAG shines.

In [ ]:
ask("What are the top themes in this story?", method="global")

### 8b — Local search: a question about specific entities
Starts from the entities that match ("Scrooge") and walks their connections.

In [ ]:
ask("Who is Scrooge and what are his main relationships?", method="local")

### 8c — Compare against plain RAG (the baseline)
`basic` is ordinary vector-similarity RAG. Ask the *same* big-picture question and compare it with 8a — this comparison is the whole point of GraphRAG.

In [ ]:
ask("What are the top themes in this story?", method="basic")

### 8d — Your turn
Try your own questions and switch methods. Ideas: *"How does Scrooge change over the course of the story?"* (global or drift), *"What is the relationship between Bob Cratchit and Tiny Tim?"* (local).

In [ ]:
ask("How does Scrooge change over the course of the story?", method="drift")

## Step 9 — Look inside the knowledge graph with pandas

Everything GraphRAG built is just **tables**. This is where it gets interesting for a data person.

*Python refresher:* `pd.read_parquet` loads a parquet file into a DataFrame; `.head(n)` shows the first n rows; `.columns` lists the column names.

> Column names can shift slightly between GraphRAG versions. If you get a `KeyError`, look at the printed `columns` and adjust.

In [ ]:
import pandas as pd

OUT = PROJECT / "output"
print("Files produced by the index:")
for p in sorted(OUT.glob("*.parquet")):
    print("  ", p.name)

entities = pd.read_parquet(OUT / "entities.parquet")
relationships = pd.read_parquet(OUT / "relationships.parquet")
reports = pd.read_parquet(OUT / "community_reports.parquet")

print(f"\nEntities: {len(entities)} | Relationships: {len(relationships)} | Community reports: {len(reports)}")
print("\nentities columns:", list(entities.columns))

### 9a — What entities did the LLM find, and of what types?

In [ ]:
pd.set_option("display.max_colwidth", 120)

print(entities["type"].value_counts(), "\n")
entities[["title", "type", "description"]].head(10)

### 9b — The most connected entities
`degree` is how many relationships an entity has — a quick proxy for importance in the story.

In [ ]:
cols = [c for c in ["title", "type", "degree"] if c in entities.columns]
entities.sort_values("degree", ascending=False)[cols].head(15)

### 9c — Relationships: who is linked to whom, and why

In [ ]:
rel_cols = [c for c in ["source", "target", "weight", "description"] if c in relationships.columns]
relationships.sort_values("weight", ascending=False)[rel_cols].head(10)

### 9d — Community reports: the summaries global search uses
Each row is one cluster of related entities, with an LLM-written title, summary and importance rating. `level` 0 is the most zoomed-out.

In [ ]:
rep_cols = [c for c in ["level", "title", "rank", "summary"] if c in reports.columns]
reports.sort_values(["level", "rank"], ascending=[True, False])[rep_cols].head(10)

## Step 10 — Draw the graph

We load the relationships into **networkx** (Python's standard graph library) and draw the 40 best-connected entities so the picture stays readable.

*Python refresher:* `nx.from_pandas_edgelist` turns a DataFrame with `source`/`target` columns straight into a graph. `G.subgraph(nodes)` keeps only the nodes you list.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.from_pandas_edgelist(relationships, source="source", target="target",
                            edge_attr="weight" if "weight" in relationships else None)

top = sorted(G.degree, key=lambda pair: pair[1], reverse=True)[:40]
H = G.subgraph([name for name, _ in top])

pos = nx.spring_layout(H, seed=42, k=0.6)  # seed = same layout every run
sizes = [150 + 120 * H.degree(n) for n in H.nodes]

plt.figure(figsize=(14, 10))
nx.draw_networkx_edges(H, pos, alpha=0.3)
nx.draw_networkx_nodes(H, pos, node_size=sizes, alpha=0.85)
nx.draw_networkx_labels(H, pos, font_size=8)
plt.title("A Christmas Carol — top 40 entities in the GraphRAG knowledge graph")
plt.axis("off")
plt.tight_layout()
plt.show()

## Step 11 (optional) — Export for Gephi

[Gephi](https://gephi.org) is a free desktop tool for exploring big graphs interactively. This writes the *full* graph to a `.graphml` file it can open.

In [ ]:
graphml = PROJECT / "christmas_carol.graphml"
nx.write_graphml(G, graphml)
print("Saved", graphml)

## Appendix A — Running without cloud APIs

**Option 1: the fast index method** (Step 7, `--method fast`) — far fewer LLM calls.

**Option 2: local models via Ollama.** GraphRAG calls models through **LiteLLM**, so in principle you can point it at Ollama. The catch: the model must *reliably* return structured JSON, and Microsoft notes malformed JSON is a common failure with proxied/local models — small local models often break the extraction step. Treat this as a starting point to tweak, not a guaranteed recipe. In `settings.yaml`, edit the model blocks along these lines:

```yaml
completion_models:
  default_completion_model:
    model_provider: ollama
    model: llama3.1
    api_base: http://localhost:11434

embedding_models:
  default_embedding_model:
    model_provider: ollama
    model: nomic-embed-text
    api_base: http://localhost:11434
```

## Appendix B — Where to go next

1. **Your own data:** clear out `input/`, drop in a few of your own `.txt` files, re-run Step 7. Start small.
2. **Domain entity types:** edit `entity_types` in `settings.yaml` (e.g. `[organisation, person, location, threat_actor, campaign]`).
3. **Prompt tuning:** `graphrag prompt-tune` rewrites the extraction prompts for *your* data — one of the biggest quality improvements available.

## Sources
- GraphRAG — Getting Started: https://microsoft.github.io/graphrag/get_started
- GraphRAG — CLI reference: https://microsoft.github.io/graphrag/cli/
- GraphRAG — Language model selection: https://microsoft.github.io/graphrag/config/models/
- Paper — *From Local to Global: A Graph RAG Approach to Query-Focused Summarization*: https://arxiv.org/abs/2404.16130